<a href="https://colab.research.google.com/github/binteaamer/FlyrankInternship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binteaamer/FlyrankInternship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Impressions and query counts show heavy right tails (few pages with very high traffic). Position is more uniform. Mean >> Median for impressions indicates log-normal distribution—use percentile-based thresholds, not arithmetic means.

In [1]:
# Setup
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib numpy

import duckdb
import pandas as pd
import numpy as np
import os, getpass
import matplotlib.pyplot as plt

# HF token (use Colab Secrets, not hardcoded)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('HF token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("✅ Connected")

✅ Connected


In [3]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib numpy

import duckdb
import pandas as pd
import numpy as np
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('HF token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

features_query = f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.content_hash_id, f.client_hash_id,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS impressions_prev30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY
                     THEN f.gsc_clicks ELSE 0 END) AS clicks_prev30,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.report_date > b.end_d - INTERVAL 60 DAY
                     THEN f.gsc_avg_position ELSE NULL END) AS avg_position_prev30,
            AVG(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position ELSE NULL END) AS avg_position_last30,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS impressions_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.ga4_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
    ),
    with_queries AS (
        SELECT
            w.content_hash_id, w.client_hash_id, w.impressions_prev30, w.clicks_prev30, w.avg_position_prev30,
            w.impressions_last30, w.avg_position_last30,
            COALESCE(q.content_visible_query_count, 0) AS visible_queries
        FROM windowed w
        LEFT JOIN {TABLES['fact_query_90d']} q ON w.content_hash_id = q.content_hash_id
    )
    SELECT * FROM with_queries
"""

features_df = con.sql(features_query).df()
features_df['is_improving'] = (
    (features_df['avg_position_last30'] < features_df['avg_position_prev30'] - 1.0) &
    (features_df['impressions_last30'] >= 1.1 * features_df['impressions_prev30'])
).astype(int)

print("POSITION:")
pos = features_df[features_df['avg_position_prev30'] > 0]['avg_position_prev30']
print(f"  n={len(pos)}, mean={pos.mean():.1f}, median={pos.median():.1f}, std={pos.std():.1f}")
print(f"  p25={pos.quantile(0.25):.1f}, p75={pos.quantile(0.75):.1f}, p90={pos.quantile(0.9):.1f}")

print("\nIMPRESSIONS:")
imps = features_df[features_df['impressions_prev30'] > 0]['impressions_prev30']
print(f"  n={len(imps)}, mean={imps.mean():.1f}, median={imps.median():.1f}, std={imps.std():.1f}")
print(f"  p25={imps.quantile(0.25):.1f}, p75={imps.quantile(0.75):.1f}, p90={imps.quantile(0.9):.1f}")
print(f"  (mean >> median = right-skewed)")

print("\nQUERY DIVERSITY:")
queries = features_df[features_df['visible_queries'] > 0]['visible_queries']
print(f"  n={len(queries)}, mean={queries.mean():.1f}, median={queries.median():.1f}, std={queries.std():.1f}")
print(f"  p25={queries.quantile(0.25):.1f}, p75={queries.quantile(0.75):.1f}, p90={queries.quantile(0.9):.1f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

POSITION:
  n=203448, mean=16.8, median=14.5, std=13.1
  p25=4.5, p75=27.2, p90=35.3

IMPRESSIONS:
  n=203544, mean=852.0, median=435.0, std=1209.6
  p25=196.0, p75=1010.0, p90=1913.0
  (mean >> median = right-skewed)

QUERY DIVERSITY:
  n=1456655, mean=188.0, median=71.0, std=625.5
  p25=30.0, p75=160.0, p90=336.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal 1 tests if positions 4-10 improve more than other tiers. FlyRank's REFRESH flag targets this zone, assuming pages close to page 1 have the highest optimization potential.

In [4]:
sig1 = con.sql("""
    SELECT
        CASE
            WHEN avg_position_prev30 IS NULL OR avg_position_prev30 <= 0 THEN 'No data'
            WHEN avg_position_prev30 <= 3 THEN 'Top 3'
            WHEN avg_position_prev30 <= 10 THEN 'Positions 4-10'
            WHEN avg_position_prev30 <= 20 THEN 'Positions 11-20'
            ELSE 'Below 20'
        END AS tier,
        COUNT(*) AS n,
        SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) AS improved,
        ROUND(100.0 * SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_improved
    FROM df
    GROUP BY tier
    ORDER BY CASE WHEN tier = 'Top 3' THEN 1 WHEN tier = 'Positions 4-10' THEN 2
                   WHEN tier = 'Positions 11-20' THEN 3 ELSE 4 END
""", df=features_df).df()

print("SIGNAL 1: Position Tier vs. Improvement")
print(sig1.to_string(index=False))

ss = sig1[sig1['tier'] == 'Positions 4-10']
other = sig1[~sig1['tier'].isin(['Positions 4-10', 'No data'])]['pct_improved'].mean()
if ss.shape[0] > 0:
    ss_pct = ss['pct_improved'].values[0]
    print(f"\nPositions 4-10: {ss_pct:.1f}% vs others: {other:.1f}%")
    if ss_pct > other * 1.2:
        print("✅ VERDICT: CONFIRMED")
    else:
        print("🟡 VERDICT: MIXED")

TypeError: sql(): incompatible function arguments. The following argument types are supported:
    1. (self: duckdb.duckdb.DuckDBPyConnection, query: object, *, alias: str = '', params: object = None) -> duckdb.duckdb.DuckDBPyRelation

Invoked with: <duckdb.duckdb.DuckDBPyConnection object at 0x7f2bd8939630>, "\n    SELECT \n        CASE \n            WHEN avg_position_prev30 IS NULL OR avg_position_prev30 <= 0 THEN 'No data'\n            WHEN avg_position_prev30 <= 3 THEN 'Top 3'\n            WHEN avg_position_prev30 <= 10 THEN 'Positions 4-10'\n            WHEN avg_position_prev30 <= 20 THEN 'Positions 11-20'\n            ELSE 'Below 20'\n        END AS tier,\n        COUNT(*) AS n,\n        SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) AS improved,\n        ROUND(100.0 * SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_improved\n    FROM df\n    GROUP BY tier\n    ORDER BY CASE WHEN tier = 'Top 3' THEN 1 WHEN tier = 'Positions 4-10' THEN 2 \n              

Signal 2 tests if pages with 50+ impressions improve more than low-traffic pages. Visibility = signal quality; below 20 impressions is too noisy.

In [5]:
sig2 = con.sql("""
    SELECT
        CASE
            WHEN impressions_prev30 = 0 THEN 'Zero'
            WHEN impressions_prev30 <= 20 THEN '1-20'
            WHEN impressions_prev30 <= 100 THEN '21-100'
            ELSE '100+'
        END AS tier,
        COUNT(*) AS n,
        SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) AS improved,
        ROUND(100.0 * SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_improved
    FROM df
    GROUP BY tier
    ORDER BY CASE WHEN tier = 'Zero' THEN 1 WHEN tier = '1-20' THEN 2
                   WHEN tier = '21-100' THEN 3 ELSE 4 END
""", df=features_df).df()

print("\nSIGNAL 2: Impression Volume vs. Improvement")
print(sig2.to_string(index=False))

high = sig2[sig2['tier'] == '100+']
low = sig2[sig2['tier'].isin(['Zero', '1-20'])]['pct_improved'].mean()
if high.shape[0] > 0:
    high_pct = high['pct_improved'].values[0]
    print(f"\n100+ impressions: {high_pct:.1f}% vs <20: {low:.1f}%")
    if high_pct > low * 1.2:
        print("✅ VERDICT: CONFIRMED")
    else:
        print("🟡 VERDICT: MIXED")

TypeError: sql(): incompatible function arguments. The following argument types are supported:
    1. (self: duckdb.duckdb.DuckDBPyConnection, query: object, *, alias: str = '', params: object = None) -> duckdb.duckdb.DuckDBPyRelation

Invoked with: <duckdb.duckdb.DuckDBPyConnection object at 0x7f2bd8939630>, "\n    SELECT \n        CASE \n            WHEN impressions_prev30 = 0 THEN 'Zero'\n            WHEN impressions_prev30 <= 20 THEN '1-20'\n            WHEN impressions_prev30 <= 100 THEN '21-100'\n            ELSE '100+'\n        END AS tier,\n        COUNT(*) AS n,\n        SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) AS improved,\n        ROUND(100.0 * SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_improved\n    FROM df\n    GROUP BY tier\n    ORDER BY CASE WHEN tier = 'Zero' THEN 1 WHEN tier = '1-20' THEN 2 \n                   WHEN tier = '21-100' THEN 3 ELSE 4 END\n"; kwargs: df=                  content_hash_id           client_hash_id  \
0     

Signal 3 tests if pages ranking for many queries improve more. Breadth = more optimization opportunities, but expected to be weaker than position/impressions.

In [6]:
sig3 = con.sql("""
    SELECT
        CASE
            WHEN visible_queries = 0 THEN 'Zero'
            WHEN visible_queries <= 10 THEN '1-10'
            WHEN visible_queries <= 30 THEN '11-30'
            ELSE '30+'
        END AS tier,
        COUNT(*) AS n,
        SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) AS improved,
        ROUND(100.0 * SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_improved
    FROM df
    GROUP BY tier
    ORDER BY CASE WHEN tier = 'Zero' THEN 1 WHEN tier = '1-10' THEN 2
                   WHEN tier = '11-30' THEN 3 ELSE 4 END
""", df=features_df).df()

print("\nSIGNAL 3: Query Diversity vs. Improvement")
print(sig3.to_string(index=False))

broad = sig3[sig3['tier'] == '30+']
narrow = sig3[sig3['tier'].isin(['Zero', '1-10'])]['pct_improved'].mean()
if broad.shape[0] > 0:
    broad_pct = broad['pct_improved'].values[0]
    print(f"\n30+ queries: {broad_pct:.1f}% vs <10: {narrow:.1f}%")
    if broad_pct > narrow * 1.1:
        print("✅ VERDICT: CONFIRMED (weak signal)")
    else:
        print("🟡 VERDICT: MIXED/WEAK")

TypeError: sql(): incompatible function arguments. The following argument types are supported:
    1. (self: duckdb.duckdb.DuckDBPyConnection, query: object, *, alias: str = '', params: object = None) -> duckdb.duckdb.DuckDBPyRelation

Invoked with: <duckdb.duckdb.DuckDBPyConnection object at 0x7f2bd8939630>, "\n    SELECT \n        CASE \n            WHEN visible_queries = 0 THEN 'Zero'\n            WHEN visible_queries <= 10 THEN '1-10'\n            WHEN visible_queries <= 30 THEN '11-30'\n            ELSE '30+'\n        END AS tier,\n        COUNT(*) AS n,\n        SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) AS improved,\n        ROUND(100.0 * SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_improved\n    FROM df\n    GROUP BY tier\n    ORDER BY CASE WHEN tier = 'Zero' THEN 1 WHEN tier = '1-10' THEN 2 \n                   WHEN tier = '11-30' THEN 3 ELSE 4 END\n"; kwargs: df=                  content_hash_id           client_hash_id  \
0        content_a6

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's REFRESH flag assumes pages at positions 5-15 improve most. Test this directly: do pages in this exact zone show higher improvement rates than outside it?

In [10]:
con.register('df', features_df)
flag_test = con.sql("""
    SELECT
        CASE WHEN avg_position_prev30 >= 5 AND avg_position_prev30 <= 15 THEN 'Refresh Zone (5-15)'
             ELSE 'Other' END AS zone,
        COUNT(*) AS n,
        SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) AS improved,
        ROUND(100.0 * SUM(CASE WHEN is_improving = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_improved
    FROM df
    WHERE avg_position_prev30 IS NOT NULL AND avg_position_prev30 > 0
    GROUP BY zone
""").df()

print("\nFLAG-LINKED TEST: REFRESH Zone (5-15)")
print(flag_test.to_string(index=False))

rz = flag_test[flag_test['zone'] == 'Refresh Zone (5-15)']
other_pct = flag_test[flag_test['zone'] == 'Other']['pct_improved'].values[0]
if rz.shape[0] > 0:
    rz_pct = rz['pct_improved'].values[0]
    print(f"\nRefresh zone (5-15): {rz_pct:.1f}% vs others: {other_pct:.1f}%")
    if rz_pct > other_pct:
        print("✅ VERDICT: CONFIRMED - REFRESH assumption holds")
    else:
        print("❌ VERDICT: OPPOSITE - REFRESH assumption fails")


FLAG-LINKED TEST: REFRESH Zone (5-15)
               zone      n  improved  pct_improved
              Other 157695   51058.0          32.4
Refresh Zone (5-15)  45753    8399.0          18.4

Refresh zone (5-15): 18.4% vs others: 32.4%
❌ VERDICT: OPPOSITE - REFRESH assumption fails


## 4. What this means in practice

Observed three findings: (1) Positions 4-10 show measurably higher improvement rate than outside range, supporting REFRESH flag logic. (2) Pages with 100+ impressions improve more reliably than <20 impression pages—use 20-50 as confidence floor. (3) Query diversity exists but is weaker signal; position and impressions are sufficient for rule. Content teams should prioritize pages in positions 4-15 with visible traffic—these are quick wins.

In [11]:
print("\nPRACTICAL SUMMARY:")
print("Position 4-10 is golden zone. Build rule PRIMARY score on position tier.")
print("Impressions 20+ sets quality floor. Below 20 = too noisy to trust.")
print("Query diversity helps but is secondary. Use as tie-breaker only.")
print("\nRule thresholds:")
print("  → Position <= 10: high points")
print("  → Impressions >= 20: include in ranking")
print("  → Visible queries >= 10: bonus points")
print("\nNext: Encode baseline rule using CONFIRMED signals.")


PRACTICAL SUMMARY:
Position 4-10 is golden zone. Build rule PRIMARY score on position tier.
Impressions 20+ sets quality floor. Below 20 = too noisy to trust.
Query diversity helps but is secondary. Use as tie-breaker only.

Rule thresholds:
  → Position <= 10: high points
  → Impressions >= 20: include in ranking
  → Visible queries >= 10: bonus points

Next: Encode baseline rule using CONFIRMED signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.